# Demo v3 — the paper-aligned scenario grid

This notebook produces **every result file the v3 demonstration reads**, by running the
paper-grid scenarios of `.specs/demo-v3/implementation.md` §0.1 through the real
optimisation model, one step at a time.

It replaces the three-scenario walkthrough of the previous version. Instead of
`S1_essential` / `S2_balanced` / `S3_ambitious` (which varied budget only, at the
demo's own operational ratio and the observed Geneva rhythm), the demonstration is now
built on **18 runs of one grid**: the paper's own baseline (Table D.8: 80 000 €,
operational ratio 5 %, T = 3, ε = 0.04) with one parameter swept at a time along four
axes — investment budget, operational ratio, dispatch-penalty ε, and the demand's
temporal profile. Every difference between two runs is therefore attributable to
exactly one change, which is what lets the demonstration's four questions (below) be
answered with a chart instead of an anecdote.

**What running this notebook top to bottom produces:**

| Step | Produces |
|---|---|
| 0 — prepare the process | nothing written; a Gurobi licence check |
| 1 — derived data | `data/calibration.json`, `data/profiles.json`, `data/pt_ridership_summary.json`, `data/stations_real.geojson` |
| 2 — the scenario grid | `scenarios/<id>.json`, one per selected scenario (generated from `run_model.PAPER_GRID`) |
| 3 — run the grid | `results/<id>/{stations,metrics,instance,model_plan}.json` per selected scenario, `results/shared/bike_arcs.json` — **the only step that calls Gurobi** |
| 4 — stress test | `results/<id>/{sim_monday,sim_sunday}.json` per selected scenario (observed-trip replay) |
| 5 — evaluate | `results/<id>/kpis.json` per selected scenario (schema `kpis-v3`) and family comparison tables |
| 6 — the four questions | one chart each, read from the KPIs above |
| 7 — what was written | an inventory of the results directory, for a sanity check |
| 8 — clean-up | guarded, off by default: removes the legacy `S1/S2/S3` runs once the grid is complete |

**Which scenarios:** the `SCENARIOS` cell right below selects what steps 2–5 (re)generate —
the whole 18-run grid by default, or a list of ids such as `["budget_020k"]` to redo one.
Steps 6–8 always read whatever is on disk for the whole grid.

**Two rules govern every cell below:**

1. **The frozen model is never edited.** Everything under
   [`network-design-bss/src/`](../network-design-bss/src) stays exactly as submitted;
   scenario parameters are injected at runtime (`run_model.scenario_parameters`), never
   by editing `instance_builder.py` on disk.
2. **Every artefact carries provenance.** Each JSON this notebook writes has a `run`
   block recording the parameters, the solver status, the wall clock and the git commits
   of both this repository and `network-design-bss/src/` — so any number in the public
   demonstration can be traced back to the exact code and settings that produced it.

> **Gurobi.** Step 3 is the only step that solves anything; it is real optimiser output,
> not a mock. Steps 0–2 and 4–8 are stdlib/pandas/matplotlib only and safe to re-run at
> any time.


## Which scenarios to (re)generate

The cell below is the only thing to edit for a partial run. Steps 2 to 5 (scenario files,
model runs, stress test, KPIs) all work on the same selection; steps 6 to 8 read whatever
is on disk for the whole grid, so a partial run still refreshes the charts and the
inventory. Ids are checked against `run_model.PAPER_GRID` in step 2 — a typo fails there,
before anything is written.


In [ ]:
# Scenario ids to (re)generate. Steps 2 (scenario files), 3 (model runs), 4 (stress test)
# and 5 (KPIs) all work on this selection.
#   SCENARIOS = None                              # the whole paper grid (default, 18 runs)
#   SCENARIOS = ["budget_020k"]                   # only this one
#   SCENARIOS = ["budget_020k", "budget_040k"]    # a few
SCENARIOS = None

# Ids to leave out of the selection, e.g. SKIP = {"eps_001"} (see step 3 for why).
SKIP = set()

# Step 3 is idempotent by default: a scenario whose results/<id>/metrics.json already
# exists is not re-solved. Set RERUN = True to re-solve the selection anyway -- so
# "regenerate budget_020k" is SCENARIOS = ["budget_020k"] together with RERUN = True.
RERUN = False

# Step 2 keeps existing scenarios/<id>.json files (their copy may be hand-edited).
# Set OVERWRITE_SCENARIO_FILES = True to rewrite the selection from PAPER_GRID.
OVERWRITE_SCENARIO_FILES = False


## Step 0 — Prepare the process

[`network-design-bss/src/`](../network-design-bss/src) is Zhenyu WU's optimisation model,
treated as **read-only** so its results stay attributable to the code as submitted. Two
things follow from that, and both are handled for you:

- `compat.bootstrap()` (in the SDK, outside the frozen tree) puts the model on `sys.path`,
  changes the working directory into it, and patches four defects at runtime.
- The scenario parameters are **not** applied by editing `instance_builder.py`, as
  `scenarios/README.md` describes. [`run_model.py`](../demo/experiments/run_model.py)
  swaps `generate_h3_instances()` in `sys.modules` for the duration of one run, so the
  file on disk is never touched and `git diff` stays empty — nothing to remember to revert.

In [ ]:
import pathlib, sys

# Tolerate re-running this cell after bootstrap() has already moved us.
REPO = pathlib.Path.cwd()
while not (REPO / "network-design-bss" / "sdk-builder").is_dir():
    if REPO.parent == REPO:
        raise FileNotFoundError("repository root not found above " + str(pathlib.Path.cwd()))
    REPO = REPO.parent

for path in (REPO, REPO / "network-design-bss" / "sdk-builder"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print("repository:", REPO)
print("python    :", sys.version.split()[0])

### The Gurobi licence

The integrated solve is a mixed-integer program well past the 2 000 variables / 2 000
constraints of the licence bundled with `gurobipy`. The cell below fails loudly now
rather than after the expensive network stage has already run.

In [ ]:
import gurobipy as gp

probe = gp.Model("licence-probe")
probe.addVars(5000, vtype="B")
probe.update()
print(f"gurobipy {'.'.join(str(v) for v in gp.gurobi.version())}"
      f" — {probe.NumVars} binary variables accepted (a size-limited licence stops at 2000)")
probe.dispose()

## Step 1 — Derived data

These four scripts turn the observed layers in
[`demo/experiments/data/geneva_1.5km-radius/`](../demo/experiments/data/geneva_1.5km-radius)
into the small derived files the simulator, the scenario grid and the KPIs need. They
depend on no scenario, so they run once, before anything else.

| Script | Produces | What it measures |
|---|---|---|
| `trips.py` | `data/calibration.json` | daily trip volumes per day type — the replay stress test's base demand |
| `profiles.py` | `data/profiles.json` | Monday/Sunday hourly profiles and the model's period weights |
| `ridership.py` | `data/pt_ridership_summary.json` | real PT boardings; cross-checks the trips' timezone reading |
| `stations_real.py` | `data/stations_real.geojson` | today's network with real GBFS capacities (map layer) |

`data/profiles.json` matters beyond the replay stress test: the **`rhythm_geneva`**
scenario of the grid (step 2) is not one of the paper's three fixed temporal profiles —
its `period_weights` are read from `profiles.json["period_weights"]["weekday"]` at
scenario-generation time (`run_model.period_weights_for`), so this cell must run before
`write_scenarios()` produces an accurate `rhythm_geneva.json`.

In [ ]:
from demo.experiments import trips, profiles, ridership, stations_real

calibration = trips.build_calibration()
profile_data = profiles.build_profiles()
pt_summary = ridership.build_summary()
real_network, match_report = stations_real.build_stations_real()

print("\nperiod weights (06-10 / 10-16 / 16-22), weekday:",
      profile_data["period_weights"]["weekday"])

The weekday period weights printed above are what `rhythm_geneva` uses as its
`model_parameters.period_weights` — the observed commute rhythm, as opposed to the
paper's synthetic bimodal / uniform / sharp profiles used by the other scenarios.

In [ ]:
from demo.experiments.ridership import validate_bike_timezone

# The trip timestamps are naive strings in a field named `trip_started_at_utc`. Reading
# them as UTC puts the morning peak at 08h local; reading them naively puts it at 06h,
# which is not credible for commuting. PT ridership is in local time by construction, so
# its peaks are an independent check on that reading.
print(validate_bike_timezone())

## Step 2 — Generate the scenario grid

The 18 scenarios are generated, not hand-written: `run_model.PAPER_GRID` is the single
definition (paper's Table D.8 baseline plus one axis swept at a time), and
`write_scenarios(ids=...)` turns the selected entries into `scenarios/<id>.json` — the
whole grid when `SCENARIOS` is `None`, only the listed ids otherwise. **Existing files are
kept unless `OVERWRITE_SCENARIO_FILES = True`** — a scenario's copy (title, pitch,
narrative) may have been hand-edited after generation, and re-running this cell must not
silently discard that. Overwrite only when `PAPER_GRID` itself changed and the copy edits
should be regenerated from scratch.

From a shell the same thing is `python -m demo.experiments.run_model --write-scenarios
[budget_020k ...] [--overwrite]`.


In [ ]:
from demo.experiments.run_model import PAPER_GRID, paper_scenarios, write_scenarios

# The whole grid (steps 6-8) and the selection this run works on (steps 2-5).
grid_ids = [s["id"] for s in paper_scenarios()]
selected_ids = [s["id"] for s in paper_scenarios(SCENARIOS) if s["id"] not in SKIP]
print(f"PAPER_GRID defines {len(grid_ids)} scenarios; this run works on {len(selected_ids)}: "
      + ", ".join(selected_ids))

written = write_scenarios(ids=selected_ids, overwrite=OVERWRITE_SCENARIO_FILES)
print(f"{len(written)} scenario file(s) (re)written; the rest already existed and were left alone")


**The four families**, each isolating one axis of the paper's sensitivity analysis
(`.specs/demo-v3/implementation.md` §0.1) around the shared baseline (`budget_080k`,
family `"baseline"`, belongs to every family's chart at its own axis value):

| family | axis | members | paper figure |
|---|---|---|---|
| `budget` | `total_budget` | 20k · 40k · 60k · **80k (baseline)** · 100k · 120k | Fig. 9, 12a, 14 — diminishing returns to investment |
| `ops_ratio` | `op_budget_ratio` | 0 % · 2.5 % · **5 % (baseline)** · 7.5 % · 10 % · 12.5 % | Fig. 7–8 — rebalancing effort vs the operational envelope |
| `epsilon` | `epsilon` (dispatch penalty) | 0 · 0.01 · **0.04 (baseline)** · 0.08 · 0.12 | Fig. 6–8, Table 3 — tractability and predictability of dispatching |
| `rhythm` | `temporal_profile` | **bimodal (baseline)** · uniform · sharp · geneva (observed) | Fig. 12 — does the daily rhythm change the layout? |

`budget_020k` and `budget_040k` sit below the paper's own budget range (`outside_paper_range: true`)
— they exist only to locate the PT-integration threshold of Fig. 14, not to reproduce a
paper figure directly. `rhythm_geneva` is similarly outside the paper's three profiles:
it is Geneva's own observed weekday rhythm, kept for a local-data comparison.

In [ ]:
import pandas as pd

scenarios = paper_scenarios()
grid_table = pd.DataFrame([
    {
        "id": s["id"],
        "family": s["family"],
        "role": s["role"],
        "card": s.get("card"),
        "axis_label": s["axis_label"],
        "temporal_profile": s["temporal_profile"],
        "total_budget": s["model_parameters"]["total_budget"],
        "op_budget_ratio": s["model_parameters"]["op_budget_ratio"],
        "epsilon": s["model_parameters"]["epsilon"],
        "period_weights": s["model_parameters"]["period_weights"],
        "outside_paper_range": s["outside_paper_range"],
        "paper_reference": s["paper_reference"],
    }
    for s in scenarios
])
grid_table

## Step 3 — Run the grid

**This is the only step that calls Gurobi.** The first scenario solved pays a cold
shortest-path enumeration (~50 min); it is cached on disk under keys that carry **no
scenario parameter**, so the other runs reuse it and take roughly 15–60 s each — the
whole grid is on the order of an hour on a cold cache, minutes on a warm one.

What this loop solves is decided by the selection cell at the top of the notebook:
`SCENARIOS` minus `SKIP`, and — with `RERUN = False`, the default — only the scenarios
that have no `results/<id>/metrics.json` yet, so re-running the notebook top to bottom
costs nothing once the grid exists. To redo one run, set `SCENARIOS = ["budget_020k"]`
and `RERUN = True` up there.

One candidate for `SKIP`: `eps_001` sits in the paper's own solver "valley" (Fig. 6) and
may run up to the model's 3600 s time limit without reaching optimality — leave it out on
a first pass and run it on its own (`SCENARIOS = ["eps_001"]`) when there is time to
wait for it.


In [ ]:
import time

from demo.experiments import RESULTS_DIR
from demo.experiments.run_model import run_scenario

print(f"running {len(selected_ids)} scenario(s): {', '.join(selected_ids)}")

run_log = []
for scenario_id in selected_ids:
    metrics_path = RESULTS_DIR / scenario_id / "metrics.json"
    if metrics_path.is_file() and not RERUN:
        print(f"[skip] {scenario_id}: results/{scenario_id}/metrics.json already exists (RERUN=False)")
        continue

    print(f"\n{'=' * 70}\n{scenario_id}\n{'=' * 70}")
    started = time.time()
    try:
        result = run_scenario(scenario_id)
        elapsed = time.time() - started
        run = result["provenance"]["run"]
        metrics = result["metrics"]
        run_log.append({
            "scenario": scenario_id,
            "status": "optimal" if run["gurobi_status_optimal"] else f"gurobi status {run['gurobi_status']}",
            "mip_gap": run["mip_gap"],
            "wall_clock_s": round(elapsed, 1),
            "stations": len(result["stations"]),
            "served_flow": (metrics.get("flow_bike_only", 0) or 0) + (metrics.get("flow_bike_pt", 0) or 0),
        })
        print(f"  done in {elapsed / 60:.1f} min: {len(result['stations'])} stations, "
              f"gurobi status {run['gurobi_status']}, gap {run['mip_gap']:.2%}")
    except Exception as exc:
        elapsed = time.time() - started
        print(f"  FAILED after {elapsed / 60:.1f} min: {exc!r}")
        run_log.append({
            "scenario": scenario_id, "status": f"ERROR: {exc}", "mip_gap": None,
            "wall_clock_s": round(elapsed, 1), "stations": None, "served_flow": None,
        })

In [ ]:
pd.DataFrame(run_log)

## Step 4 — Stress test with observed trips

The primary evaluation (step 5) is the model's own solution, exactly as the paper
reports it. This step adds a **demo-side robustness check**: replay Geneva's actually
recorded trips (~11.5/day Monday, ~9.4/day Sunday — about 125× smaller than the
1 453-trip planning day the designs were sized for) against each finished design and
record what fails and why (no station nearby, no bike, no dock). It is not part of the
paper and is not used to rank plans; it answers a different question — "does this
design also cope with the traffic Geneva has today?"

Every selected scenario that has a `stations.json` (i.e. was solved in step 3, in this
run or an earlier one) is replayed for both day types.

In [ ]:
from demo.experiments.simulate import simulate_day

for scenario_id in selected_ids:
    out_dir = RESULTS_DIR / scenario_id
    stations_file = out_dir / "stations.json"
    if not stations_file.is_file():
        print(f"[skip] {scenario_id}: no stations.json yet (not solved)")
        continue
    for day in ("monday", "sunday"):
        simulate_day(stations_file, day, mode="replay",
                     output_file=out_dir / f"sim_{day}.json")

## Step 5 — Evaluate

`evaluate_scenario(dir)` reads `model_plan.json` and `metrics.json` (both required) plus
`sim_monday.json` / `sim_sunday.json` when present, and writes `kpis.json`
(`"schema": "kpis-v3"`) with three blocks: `paper` (the paper's own headline numbers),
`technical` (the full evaluator row, for the advanced view) and `stress_test` (step 4's
replay, when available). Nothing here re-simulates or re-prices what the frozen model
already decided.

In [ ]:
from demo.experiments.evaluate import evaluate_scenario

evaluated_ids = []
for scenario_id in selected_ids:
    out_dir = RESULTS_DIR / scenario_id
    if not (out_dir / "model_plan.json").is_file() or not (out_dir / "metrics.json").is_file():
        print(f"[skip] {scenario_id}: model_plan.json or metrics.json missing (not solved)")
        continue
    evaluate_scenario(out_dir)
    evaluated_ids.append(scenario_id)

print(f"\n{len(evaluated_ids)}/{len(selected_ids)} selected scenarios evaluated")

In [ ]:
from IPython.display import Markdown, display

from demo.experiments.evaluate import compare

baseline_id = next(s["id"] for s in scenarios if s["family"] == "baseline")
families = ("budget", "ops_ratio", "epsilon", "rhythm")

for family in families:
    family_ids = [s["id"] for s in scenarios if s["family"] == family] + [baseline_id]
    family_dirs = [RESULTS_DIR / sid for sid in family_ids
                   if (RESULTS_DIR / sid / "kpis.json").is_file()]
    if not family_dirs:
        print(f"[skip] {family}: no evaluated scenario yet")
        continue
    display(Markdown(f"### `{family}` family\n\n" + compare(family_dirs)))

## Step 6 — The four questions

Step 1 of the public demonstration asks four questions; this step answers each with one
chart, read from the `paper` block of `kpis.json` and the `model_parameters` of the
matching `scenarios/<id>.json`. A scenario missing either file (not yet run or not yet
evaluated) is skipped with a printed note rather than breaking the chart.

In [ ]:
import json

import matplotlib.pyplot as plt

from demo.experiments.run_model import SCENARIOS_DIR


def load_kpis(scenario_id):
    """The parsed kpis.json of one scenario, or None if not yet evaluated."""
    path = RESULTS_DIR / scenario_id / "kpis.json"
    if not path.is_file():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_model_parameters(scenario_id):
    """The model_parameters block of one scenario's definition, or None."""
    path = SCENARIOS_DIR / f"{scenario_id}.json"
    if not path.is_file():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)["model_parameters"]


def question_points(scenario_ids, axis_key):
    """(axis value, paper KPIs, id) for every scenario with both files present,
    sorted by axis value; scenarios missing either are skipped and reported."""
    points = []
    for scenario_id in scenario_ids:
        kpis = load_kpis(scenario_id)
        params = load_model_parameters(scenario_id)
        if kpis is None or params is None:
            print(f"  [skip] {scenario_id}: kpis.json or scenario definition missing")
            continue
        points.append((params[axis_key], kpis["paper"], scenario_id))
    return sorted(points, key=lambda p: p[0])

### Q1 — How much should the city invest, and where does the next euro stop paying off?

Served ratio and investment per served trip, across the `budget` family plus the
baseline. The legacy `S1_essential` / `S2_balanced` / `S3_ambitious` runs (2.5 % / 2.5 %
/ 5 % operational ratio, the observed Geneva rhythm rather than the paper's bimodal
profile) are drawn as hollow markers if their results are still on disk — for reference
only, not part of the paper-aligned budget axis.

In [ ]:
budget_ids = [s["id"] for s in scenarios if s["family"] in ("budget", "baseline")]
legacy_ids = [sid for sid in ("S1_essential", "S2_balanced", "S3_ambitious")
             if (RESULTS_DIR / sid / "kpis.json").is_file()]

points = question_points(budget_ids, "total_budget")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot([p[0] for p in points], [p[1]["served_ratio"] for p in points],
        "o-", color="tab:blue", label="paper grid")
for sid in legacy_ids:
    kpis, params = load_kpis(sid), load_model_parameters(sid)
    ax1.plot(params["total_budget"], kpis["paper"]["served_ratio"],
             "o", mfc="none", mec="tab:blue", label=f"{sid} (legacy)")
ax1.set_xlabel("total budget (EUR)")
ax1.set_ylabel("served ratio")
ax1.set_title("served demand vs budget")
ax1.legend(fontsize=8)

ax2.plot([p[0] for p in points], [p[1]["investment_per_served_trip_eur"] for p in points],
        "o-", color="tab:orange")
ax2.set_xlabel("total budget (EUR)")
ax2.set_ylabel("EUR per served trip")
ax2.set_title("investment per served trip")

plt.tight_layout()
plt.show()

The paper reports **diminishing returns to infrastructure investment** (Fig. 9, 12a):
the first euros buy coverage cheaply, and each additional euro past the reference plan
buys fewer newly served trips than the one before it — the served-ratio curve should
flatten while the cost-per-served-trip curve rises through the grid's upper budgets.

### Q2 — When does bike sharing start feeding the trams and buses, instead of replacing short rides?

The share of served trips that combine a bike leg with public transport, across the same
budget axis.

In [ ]:
points = question_points(budget_ids, "total_budget")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot([p[0] for p in points], [p[1]["pt_assisted_share"] for p in points],
       "o-", color="tab:green", label="paper grid")
for sid in legacy_ids:
    kpis, params = load_kpis(sid), load_model_parameters(sid)
    ax.plot(params["total_budget"], kpis["paper"]["pt_assisted_share"],
            "o", mfc="none", mec="tab:green", label=f"{sid} (legacy)")
ax.set_xlabel("total budget (EUR)")
ax.set_ylabel("PT-assisted share of served trips")
ax.set_title("bike-only vs bike+PT regime")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

The paper finds a **budget threshold between a bike-only regime and genuine PT
integration** (Fig. 14): below it the network mostly substitutes for short walks and
rides, above it a rising share of served trips actively feed the tram and bus network —
the PT-assisted share should show a step, not a flat line, somewhere across this axis.

### Q3 — Should money go into trucks that move bikes around, or into more docks and stations?

Dispatch cost and served ratio, across the `ops_ratio` family (operational budget as a
share of the total) and the `epsilon` family (the dispatch penalty), each plus the
baseline.

In [ ]:
ops_ids = [s["id"] for s in scenarios if s["family"] in ("ops_ratio", "baseline")]
eps_ids = [s["id"] for s in scenarios if s["family"] in ("epsilon", "baseline")]

ops_points = question_points(ops_ids, "op_budget_ratio")
eps_points = question_points(eps_ids, "epsilon")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

axes[0, 0].plot([p[0] for p in ops_points], [p[1]["dispatch_cost_eur"] for p in ops_points],
               "o-", color="tab:red")
axes[0, 0].set_xlabel("operational budget ratio")
axes[0, 0].set_ylabel("dispatch cost (EUR/day)")
axes[0, 0].set_title("dispatch cost vs operational ratio")

axes[0, 1].plot([p[0] for p in ops_points], [p[1]["served_ratio"] for p in ops_points],
               "o-", color="tab:blue")
axes[0, 1].set_xlabel("operational budget ratio")
axes[0, 1].set_ylabel("served ratio")
axes[0, 1].set_title("served demand vs operational ratio")

axes[1, 0].plot([p[0] for p in eps_points], [p[1]["dispatch_cost_eur"] for p in eps_points],
               "o-", color="tab:red")
axes[1, 0].set_xlabel("epsilon (dispatch penalty)")
axes[1, 0].set_ylabel("dispatch cost (EUR/day)")
axes[1, 0].set_title("dispatch cost vs epsilon")

axes[1, 1].plot([p[0] for p in eps_points], [p[1]["served_ratio"] for p in eps_points],
               "o-", color="tab:blue")
axes[1, 1].set_xlabel("epsilon (dispatch penalty)")
axes[1, 1].set_ylabel("served ratio")
axes[1, 1].set_title("served demand vs epsilon")

plt.tight_layout()
plt.show()

The paper finds that **rebalancing contributes directly to served demand, and a small
penalty makes it predictable at almost no cost**: a modest ε cuts dispatch cost sharply
for well under 1 % of served demand (Fig. 7–8), and transfer stations at PT stops act as
connectivity buffers that reduce how much trucking is needed in the first place
(Fig. 13). Served ratio should stay close to flat across both axes while dispatch cost
moves — the model barely needs trucks when the layout is right.

### Q4 — Does the rhythm of the day change where the stations should go?

Served ratio and station compactness (nearest-neighbour distance), across the `rhythm`
family plus the baseline (each temporal profile re-optimised, not just re-simulated).

In [ ]:
rhythm_ids = [s["id"] for s in scenarios if s["family"] in ("rhythm", "baseline")]

rhythm_rows = []
for scenario_id in rhythm_ids:
    kpis = load_kpis(scenario_id)
    params = load_model_parameters(scenario_id)
    if kpis is None or params is None:
        print(f"  [skip] {scenario_id}: kpis.json or scenario definition missing")
        continue
    rhythm_rows.append((params["temporal_profile"], kpis["paper"]["served_ratio"],
                        kpis["paper"]["nearest_neighbor_m"], scenario_id))

labels = [row[0] for row in rhythm_rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(labels, [row[1] for row in rhythm_rows], color="tab:blue")
ax1.set_ylabel("served ratio")
ax1.set_title("served demand by temporal profile")
ax1.tick_params(axis="x", rotation=20)

ax2.bar(labels, [row[2] for row in rhythm_rows], color="tab:purple")
ax2.set_ylabel("nearest-neighbour distance (m)")
ax2.set_title("station compactness by temporal profile")
ax2.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

The paper finds that **temporal structure shapes both the layout and the achievable
served demand** (Fig. 12): a concentrated (sharp or leisure-like) rhythm lets the
optimiser cluster stations more tightly, while a flatter day spreads demand and tends to
serve more of it overall — served ratio and compactness should both move across the
profiles, not stay flat.

## Step 7 — What was written

An inventory of `results/`, as a sanity check before publishing: which files each
scenario directory has, what schema its `kpis.json` declares, whether it has a
`model_plan.json` (needed for the paper KPIs) and a non-empty `technical` block (the
full evaluator row, for the advanced view).

In [ ]:
all_ids = grid_ids + [sid for sid in ("S1_essential", "S2_balanced", "S3_ambitious")
                      if (RESULTS_DIR / sid).is_dir()]

inventory_rows = []
for scenario_id in all_ids:
    out_dir = RESULTS_DIR / scenario_id
    if not out_dir.is_dir():
        continue
    files = sorted(p.name for p in out_dir.iterdir() if p.is_file())
    kpis_path = out_dir / "kpis.json"
    schema = has_technical = None
    if kpis_path.is_file():
        with open(kpis_path, encoding="utf-8") as f:
            kpis_doc = json.load(f)
        schema = kpis_doc.get("schema")
        has_technical = bool(kpis_doc.get("technical"))
    inventory_rows.append({
        "scenario": scenario_id,
        "files": ", ".join(files),
        "kpis_schema": schema,
        "has_model_plan": (out_dir / "model_plan.json").is_file(),
        "has_technical_row": has_technical,
    })

inventory = pd.DataFrame(inventory_rows)
inventory

In [ ]:
bike_arcs_path = RESULTS_DIR / "shared" / "bike_arcs.json"
print(f"results/shared/bike_arcs.json present: {bike_arcs_path.is_file()}")
print("  the routed ride network (OSM distance/time between candidate stations); "
      "scenario-independent for one instance, so it is written once by the first "
      "scenario solved and shared by every scenario since -- not per-scenario.")

### Reproducibility

The frozen model must stay unmodified (`AGENTS.md` rule 1): the diff below should be
empty. Each result also carries the git commit of `network-design-bss/src/` and of this
repository at the time it was solved, so any number in the demonstration can be traced
back to the code that produced it.

In [ ]:
import subprocess

src_diff = subprocess.run(
    ["git", "diff", "--stat", "--", "network-design-bss/src"],
    cwd=REPO, capture_output=True, text=True,
).stdout
print("git diff -- network-design-bss/src (must be empty: the model is frozen):")
print(src_diff or "  (empty)")

print("\nprovenance per result (run.src_commit / run.repo_commit / run.ran_at):")
for scenario_id in all_ids:
    metrics_path = RESULTS_DIR / scenario_id / "metrics.json"
    if not metrics_path.is_file():
        continue
    with open(metrics_path, encoding="utf-8") as f:
        run = json.load(f).get("run", {})
    print(f"  {scenario_id:<16} src={run.get('src_commit')}  "
          f"repo={run.get('repo_commit')}  ran_at={run.get('ran_at')}")

## Step 8 — Clean-up of the legacy runs

`S1_essential`, `S2_balanced` and `S3_ambitious` are the demo v2 scenarios: the
observed Geneva weekday rhythm instead of the paper's bimodal profile, and (for S1/S2)
an operational ratio of 2.5 % instead of the paper's baseline 5 %. Their
`scenarios/*.json` carry `"legacy": true` and their results are kept, side by side with
the grid, **until the grid exists** — so the public demonstration keeps building on real
data during the transition (`.specs/demo-v3/implementation.md` §0.2).

The cell below is guarded and does nothing by default. Once every grid scenario has a
`kpis.json` (step 5 complete for all 18), set `REMOVE_LEGACY = True` and re-run it to
delete `results/S1_essential`, `results/S2_balanced`, `results/S3_ambitious` and their
`scenarios/*.json`.

This does **not** remove the deprecated *code* the paper grid replaces — `simulate.py`'s
`sample` / `instance` modes, `baseline.py`, `pipeline/instance.py`, the legacy KPI
families in `evaluate.py`, the deprecated blocks of `kpi_config.json`. That clean-up is a
later, separate pass, once the user has validated these runs
(`.specs/demo-v3/plan.md` §1, phase P6) — no code is touched here.

In [ ]:
REMOVE_LEGACY = False

LEGACY_IDS = ("S1_essential", "S2_balanced", "S3_ambitious")
grid_complete = all((RESULTS_DIR / sid / "kpis.json").is_file() for sid in grid_ids)

if not REMOVE_LEGACY:
    print("REMOVE_LEGACY is False -- nothing removed.")
elif not grid_complete:
    missing = [sid for sid in grid_ids if not (RESULTS_DIR / sid / "kpis.json").is_file()]
    print(f"Not removing: {len(missing)} grid scenario(s) still missing a kpis.json: "
          + ", ".join(missing))
else:
    import shutil

    for scenario_id in LEGACY_IDS:
        result_dir = RESULTS_DIR / scenario_id
        scenario_file = SCENARIOS_DIR / f"{scenario_id}.json"
        if result_dir.is_dir():
            shutil.rmtree(result_dir)
            print(f"removed {result_dir}")
        if scenario_file.is_file():
            scenario_file.unlink()
            print(f"removed {scenario_file}")